# Training base expert on vanilla OGBench environment using BC (humlarge v3)

In [1]:
import random
import torch
import os
import math

import matplotlib.pyplot as plt

from collections import defaultdict

from causal_gym import HumanoidMazeV2PCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *

<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.
/home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
num_steps = 2000
seed = 0
hidden_dims = {'W'}

random.seed(seed)
torch.manual_seed(seed)

In [4]:
env = HumanoidMazeV2PCH(num_steps=num_steps, custom_hidden=hidden_dims, expert_mode=True, seed=seed, env_id='humanoidmaze-large-navigate-singletask-task1-v0', success_radius=20.0)
train_eps = env.expert.num_eps
train_eps

1099

In [5]:
X = {f'X{t}' for t in range(num_steps)}
Y = f'Y{num_steps}'
obs_prefix = env.env.observed_unobserved_vars[0]

In [6]:
Z_sets = {}
for Xi in X:
    i = int(Xi[1:])
    cond = set()

    for j in range(i+1):
        cond.update({f'{o}{j}' for o in list(set(obs_prefix) - {'X'})})

    for j in range(i):
        cond.add(f'X{j}')

    Z_sets[Xi] = cond

Z_sets['X1']

{'A0',
 'A1',
 'C0',
 'C1',
 'E0',
 'E1',
 'H0',
 'H1',
 'J0',
 'J1',
 'P0',
 'P1',
 'R0',
 'R1',
 'V0',
 'V1',
 'X0'}

In [7]:
records = collect_expert_trajectories(
    env,
    num_episodes=train_eps,
    max_steps=num_steps,
    seed=seed,
    show_progress=True
)

Starting episode 1/1099...


  Episode 1 ended at step 2000 (terminated: 1.0, truncated: False).
Starting episode 2/1099...


  Episode 2 ended at step 2000 (terminated: 1.0, truncated: False).
Starting episode 3/1099...


  Episode 3 ended at step 2000 (terminated: 1.0, truncated: False).
Starting episode 4/1099...


  Episode 4 ended at step 2000 (terminated: 1.0, truncated: False).
Starting episode 5/1099...


...

  Episode 1095 ended at step 2000 (terminated: 1.0, truncated: False).
Starting episode 1096/1099...


  Episode 1096 ended at step 2000 (terminated: 1.0, truncated: False).
Starting episode 1097/1099...


  Episode 1097 ended at step 2000 (terminated: 1.0, truncated: False).
Starting episode 1098/1099...


  Episode 1098 ended at step 2000 (terminated: 1.0, truncated: False).
Starting episode 1099/1099...


  Episode 1099 ended at step 2000 (terminated: 1.0, truncated: False).
Finished collecting expert trajectories.


In [8]:
hidden_size = 256
lr = 3e-4
batch_size = 2048
patience = 20
lookback = 10
num_blocks = 4
epochs = 300
dropout = 0.0

dims = {
    'P': 2,
    'A': 21,
    'H': 1,
    'E': 12,
    'V': 3,
    'C': 3,
    'R': 6,
    'J': 27,
    'X': 21
}

In [9]:
model, slots, Z_trim = train_single_policy_long_horizon(
    records,
    Z_sets,
    dims=dims,
    epochs=epochs,
    include_vars=obs_prefix,
    lookback=lookback,
    continuous=True,
    num_actions = env.action_space.shape[0],
    hidden_dim=hidden_size,
    num_blocks=num_blocks,
    dropout=dropout,
    lr=lr,
    batch_size=batch_size,
    patience=patience,
    device=device,
    seed=seed,
    action_bounds=(env.action_space.low, env.action_space.high)
)

policy = shared_policy_fn_long_horizon(model, slots, Z_trim, continuous=True, device=device)
policies = make_shared_policy_dict(policy)

[LongHorizon] Epoch 1: train loss = 0.089394, val loss = 0.071901.


[LongHorizon] Epoch 2: train loss = 0.068021, val loss = 0.065256.


[LongHorizon] Epoch 3: train loss = 0.063256, val loss = 0.061834.


[LongHorizon] Epoch 4: train loss = 0.060169, val loss = 0.059372.


[LongHorizon] Epoch 5: train loss = 0.057862, val loss = 0.057309.


...

[LongHorizon] Epoch 296: train loss = 0.036346, val loss = 0.039942.


[LongHorizon] Epoch 297: train loss = 0.036335, val loss = 0.039949.


[LongHorizon] Epoch 298: train loss = 0.036317, val loss = 0.039911.


[LongHorizon] Epoch 299: train loss = 0.036318, val loss = 0.039984.


[LongHorizon] Epoch 300: train loss = 0.036303, val loss = 0.039933.


In [10]:
expert_episode_rewards = defaultdict(float)
for rec in records:
    ep = rec['episode']
    expert_episode_rewards[ep] += float(rec['reward'])

num_eps = len(expert_episode_rewards)
expert_rewards = [expert_episode_rewards[e] for e in range(num_eps)]

num_eval_eps = 20

policy_records = collect_imitator_trajectories(
    env=env,
    policies=policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    hidden_dims=hidden_dims,
    show_progress=True
)

policy_episode_rewards = defaultdict(float)
for rec in policy_records:
    ep = rec['episode']
    policy_episode_rewards[ep] += float(rec['reward'])

policy_rewards = [policy_episode_rewards[e] for e in range(num_eval_eps)]

sum(expert_rewards)/num_eps, sum(policy_rewards)/num_eval_eps

Starting episode 1/20...


  Episode 1 ended at step 2000 (terminated: False, truncated: True).
Starting episode 2/20...


  Episode 2 ended at step 2000 (terminated: False, truncated: True).
Starting episode 3/20...


  Episode 3 ended at step 2000 (terminated: False, truncated: True).
Starting episode 4/20...


  Episode 4 ended at step 1591 (terminated: True, truncated: False).
Starting episode 5/20...


...

  Episode 17 ended at step 2000 (terminated: False, truncated: True).
Starting episode 18/20...


  Episode 18 ended at step 1663 (terminated: True, truncated: False).
Starting episode 19/20...


  Episode 19 ended at step 2000 (terminated: False, truncated: True).
Starting episode 20/20...


  Episode 20 ended at step 2000 (terminated: False, truncated: True).
Finished collecting imitator trajectories.


(-1281.3384895359418, -864.2815358717726)

In [11]:
# save model for fine-tuning
import os
import torch

SAVE_DIR = '/home/et2842/causal/causalrl/models'
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, 'humanoidmaze_large_expert_v3.pt')

checkpoint = {
    "state_dict": model.state_dict(),
    "slots": slots,
    "Z_trim": Z_trim,
    "dims": dims,
    "lookback": lookback,
    "continuous": True,
    "num_actions": env.action_space.shape[0],
    "hidden_dim": hidden_size,
    "num_blocks": num_blocks,
    "dropout": 0.0,
    "layernorm": True,
    "final_tanh": True,
    "action_bounds_low": env.action_space.low,
    "action_bounds_high": env.action_space.high,
    "input_dim": int(model.hidden.in_features),
}

torch.save(checkpoint, MODEL_PATH)
print("Saved expert to:", MODEL_PATH)

Saved expert to: /home/et2842/causal/causalrl/models/humanoidmaze_large_expert_v3.pt


# Fine-tuning expert on HumanoidMaze Large (humlarge v3)

In [12]:
import random
import torch
import os
import math

import matplotlib.pyplot as plt

from collections import defaultdict

from causal_gym import HumanoidMazeV2PCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *

In [13]:
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [14]:
# load model
MODEL_PATH = "/home/et2842/causal/causalrl/models/humanoidmaze_large_expert_v3.pt"
checkpoint = torch.load(MODEL_PATH, map_location=device)

# Rebuild the model with the same architecture
action_bounds = (checkpoint['action_bounds_low'], checkpoint['action_bounds_high'])

pretrained_actor = ContinuousPolicyNN(
    input_dim=checkpoint['input_dim'],
    action_dim=checkpoint['num_actions'],
    hidden_dim=checkpoint['hidden_dim'],
    num_blocks=checkpoint['num_blocks'],
    dropout=checkpoint['dropout'],
    layernorm=checkpoint['layernorm'],
    final_tanh=checkpoint['final_tanh'],
    action_bounds=action_bounds,
).to(device)

pretrained_actor.load_state_dict(checkpoint['state_dict'])
# pretrained_actor.eval()
pretrained_actor.train()

slots = checkpoint['slots']
Z_trim = checkpoint['Z_trim']
dims = checkpoint['dims']
lookback = checkpoint['lookback']

state_dim = checkpoint['input_dim']
state_dim, lookback

/tmp/ipykernel_629735/1788701766.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(MODEL_PATH, map_location=device)


(1050, 10)

In [15]:
num_steps = 2000
rl_seed_pretrain = 2014
rl_seed = 90210
hidden_dims = set() # {'W'}

env_pretrain = HumanoidMazeV2PCH(num_steps=num_steps, expert_mode=True, seed=rl_seed_pretrain, env_id='humanoidmaze-large-navigate-singletask-task1-v0', success_radius=20.0)
env_train = HumanoidMazeV2PCH(num_steps=num_steps, expert_mode=True, seed=rl_seed, env_id='humanoidmaze-large-navigate-singletask-task1-v0', success_radius=20.0)
action_dim = env_train.env.action_space.shape[0]
action_dim

21

In [16]:
def make_dense_distance_reward(
    env,
    use_delta=True,
    c=1.0,
    success_bonus=50.0,
    success_radius=20.0,
    time_penalty=0.01,
    max_steps=None,
    scale_success_by_time=False,
    success_time_alpha=0.25,
):
    goal_xy = env.env._goal_xy

    if scale_success_by_time and max_steps is None:
        raise ValueError('max_steps must be provided when scale_success_by_time=True')

    def reward_fn(obs, reward_env):
        t = len(obs["P"]) - 1

        P_curr = obs["P"][t]
        curr_xy = np.array(P_curr[:2], dtype=np.float64)
        dist_curr = np.linalg.norm(curr_xy - goal_xy)

        # Distance shaping
        if use_delta:
            if t == 0:
                r = 0.0
            else:
                P_prev = obs["P"][t - 1]
                prev_xy = np.array(P_prev[:2], dtype=np.float64)
                dist_prev = np.linalg.norm(prev_xy - goal_xy)
                r = float(c * (dist_prev - dist_curr))
        else:
            r = float(-c * dist_curr)

        # Time pressure
        r -= time_penalty

        # Success bonus
        if dist_curr <= success_radius:
            bonus = success_bonus

            # Optional mild speed bonus
            if scale_success_by_time:
                time_left_frac = max(0.0, (max_steps - t) / max_steps)
                bonus *= (1.0 + success_time_alpha * time_left_frac)

            r += bonus

        return float(r)

    return reward_fn


reward_fn = make_dense_distance_reward(
    env_train,
    success_bonus=50.0,
    success_radius=20.0,
    time_penalty=0.01,
    scale_success_by_time=False,
)

In [17]:
config = OnlineRLConfig(
    total_env_steps=2_000_000,
    start_steps=20_000,
    max_episode_steps=num_steps,
    batch_size=512,
    gamma=0.99,
    tau=0.005,
    policy_delay=2,
    actor_lr=3e-4,
    critic_lr=3e-4,
    noise_std=0.25,
    hidden_dim_q=512,
    target_policy_noise=0.2,
    target_noise_clip=0.3,
    actor_warmup_steps=100_000,
    bc_reg_lambda=2.5,
    max_grad_norm=1.0
)

In [18]:
# pretrain critics offline
replay_buffer, q1, q2, target_q1, target_q2 = pretrain_critics_offline(
    env=env_pretrain,
    pretrained_actor=pretrained_actor,
    Z_trim=Z_trim,
    slots=slots,
    state_dim=state_dim,
    action_dim=action_dim,
    config=config,
    device=device,
    num_pretrain_steps=400_000,
    pretrain_updates=200_000,
    seed=rl_seed_pretrain,
    reward_shaping_fn=reward_fn
)

In [19]:
def callback(stats: dict):
    if stats['episode'] % 1 == 0:
        print(
            f'[Episode {stats["episode"]}] '
            f'steps={stats["env_steps"]}, '
            f'return={stats["return"]:.2f}, '
            f'len={stats["length"]}, '
            f'buffer={stats["buffer_size"]}'
        )

In [20]:
fine_tuned_policy, logs = td3_fine_tune_actor(
    env=env_train,
    actor=pretrained_actor,
    Z_trim=Z_trim,
    slots=slots,
    state_dim=state_dim,
    action_dim=action_dim,
    config=config,
    device=device,
    seed=rl_seed,
    log_callback=callback,
    replay_buffer=replay_buffer,
    initial_q1=q1,
    initial_q2=q2,
    initial_target_q1=target_q1,
    initial_target_q2=target_q2,
    reward_shaping_fn=reward_fn
)

ft_pi = shared_policy_fn_long_horizon(fine_tuned_policy, slots, Z_trim, continuous=True, device=device)
ft_policies = make_shared_policy_dict(ft_pi)

[Episode 1] steps=2000, return=-12.70, len=2000, buffer=402304


[Episode 2] steps=4000, return=-13.37, len=2000, buffer=404304


[Episode 3] steps=6000, return=-14.39, len=2000, buffer=406304


[Episode 4] steps=7716, return=106.88, len=1716, buffer=408020


[Episode 5] steps=9716, return=-5.42, len=2000, buffer=410020


...

[Episode 999] steps=1993108, return=-20.71, len=2000, buffer=1000000


[Episode 1000] steps=1995108, return=-21.04, len=2000, buffer=1000000


[Episode 1001] steps=1997108, return=-20.63, len=2000, buffer=1000000


[Episode 1002] steps=1999108, return=-20.51, len=2000, buffer=1000000


[Episode 1003] steps=2001108, return=-19.80, len=2000, buffer=1000000


In [21]:
expert_env = HumanoidMazeV2PCH(num_steps=num_steps, expert_mode=True, env_id='humanoidmaze-large-navigate-singletask-task1-v0', success_radius=20.0)

In [22]:
num_eval_eps = 100

records = collect_imitator_trajectories(
    env=expert_env,
    policies=ft_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    hidden_dims=hidden_dims,
    show_progress=True
)

Starting episode 1/100...


  Episode 1 ended at step 2000 (terminated: False, truncated: True).
Starting episode 2/100...


  Episode 2 ended at step 2000 (terminated: False, truncated: True).
Starting episode 3/100...


  Episode 3 ended at step 2000 (terminated: False, truncated: True).
Starting episode 4/100...


  Episode 4 ended at step 2000 (terminated: False, truncated: True).
Starting episode 5/100...


...

  Episode 96 ended at step 585 (terminated: True, truncated: False).
Starting episode 97/100...


  Episode 97 ended at step 1224 (terminated: True, truncated: False).
Starting episode 98/100...


  Episode 98 ended at step 1640 (terminated: True, truncated: False).
Starting episode 99/100...


  Episode 99 ended at step 2000 (terminated: False, truncated: True).
Starting episode 100/100...


  Episode 100 ended at step 2000 (terminated: False, truncated: True).
Finished collecting imitator trajectories.


In [23]:
# save expert
import os
import torch

SAVE_DIR = '/home/et2842/causal/causalrl/models'
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, 'humanoidmaze_large_expert_finetuned_v3.pt')

checkpoint = {
    "state_dict": fine_tuned_policy.state_dict(),
    "slots": slots,
    "Z_trim": Z_trim,
    "dims": dims,
    "lookback": lookback,
    "continuous": True,
    "num_actions": env_train.action_space.shape[0],
    "hidden_dim": config.hidden_dim_q,
    "num_blocks": checkpoint['num_blocks'],
    "dropout": 0.0,
    "layernorm": True,
    "final_tanh": True,
    "action_bounds_low": env_train.action_space.low,
    "action_bounds_high": env_train.action_space.high,
    "input_dim": int(fine_tuned_policy.hidden.in_features),
}

torch.save(checkpoint, MODEL_PATH)
print("Saved expert to:", MODEL_PATH)

Saved expert to: /home/et2842/causal/causalrl/models/humanoidmaze_large_expert_finetuned_v3.pt
